# Aplicando técnicas de otimização de hiperparâmetros para melhorar o desempenho do modelo de predição de resistência

In [34]:
%%capture
!pip install pymoo

In [41]:
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from xgboost import XGBRegressor
from pymoo.optimize import minimize
from google.colab import drive
import pandas as pd
import numpy as np
import os


In [36]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
dirpath = '/content/drive/My Drive/supervised-learning-studies'

model_path = os.path.join(dirpath, "melhor_modelo_resistencia.json")
train_data_path = os.path.join(dirpath, 'df_train.pkl')
model = XGBRegressor()
model.load_model(model_path)
df_train = pd.read_pickle(train_data_path)
feature_names = df_train.drop(columns=['target']).columns.tolist()

In [42]:
class ResistenciaOptimization(ElementwiseProblem):

    def __init__(self, xgb_model, feature_names):
        self.xgb_model = xgb_model
        self.feature_names = feature_names
        
        # Definindo os limites (Lower e Upper Bounds) baseados no seu dataset original
        # Exemplo: cimento entre 100 e 500, água entre 150 e 250, etc.
        xl = np.array([100, 0, 0, 150, 0, 700, 600, 1]) 
        xu = np.array([550, 300, 200, 250, 30, 1100, 950, 365])

        super().__init__(n_var=8, n_obj=1, n_constr=0, xl=xl, xu=xu)

    def _evaluate(self, x, out, *args, **kwargs):
        # x é um vetor com os valores das 8 variáveis
        prediction = self.xgb_model.predict(x.reshape(1, -1))[0]
        
        # O Pymoo sempre MINIMIZA. Como queremos MAXIMIZAR a resistência, 
        # passamos o valor negativo.
        out["F"] = [-prediction]

In [ ]:
class ReceitaPorResistencia(ElementwiseProblem):

    def __init__(self, xgb_model, target_value):
        self.xgb_model = xgb_model
        self.target_value = target_value # O valor que você deseja (ex: 45.0)
        
        # Mantenha os mesmos limites que você já definiu
        xl = np.array([100, 0, 0, 150, 0, 700, 600, 1]) 
        xu = np.array([550, 300, 200, 250, 30, 1100, 950, 365])

        super().__init__(n_var=8, n_obj=1, n_constr=0, xl=xl, xu=xu)

    def _evaluate(self, x, out, *args, **kwargs):
        # x chega como um array de tamanho (8,)
        # Transformamos em (1, 8) para o modelo não reclamar do shape
        input_data = x.reshape(1, -1).astype(np.float32)
        
        # Predição
        prediction = self.xgb_model.predict(input_data)[0]
        
        # Objetivo: Minimizar a diferença absoluta (erro) para o alvo dinâmico
        erro = np.abs(prediction - self.target_value)
        
        out["F"] = erro

In [54]:
problem = ResistenciaOptimization(model, feature_names=feature_names)

algorithm = NSGA2(
    pop_size=100,
    sampling=FloatRandomSampling(),
)

res = minimize(problem,
               algorithm,
               ('n_gen', 200),
               seed=42,
               verbose=False)

melhor_solucao = res.X[0, :].reshape(1, -1)

print(f"Melhor Resistência encontrada: {-res.F[0][0]:.2f} MPa")
print(f"Ingredientes para atingir o alvo: \n\t{"\n\t".join([f'{name}: {value}' for name, value in zip(feature_names, melhor_solucao.flatten())])}")

Melhor Resistência encontrada: 84.10 MPa
Ingredientes para atingir o alvo: 
	cimento: 545.371938767087
	escoriadealtoforno: 163.08811756682982
	cinzasvolantes: 2.6199117735115074
	agua: 150.00407954312777
	superplastificante: 11.454175486480262
	agregadogrosso: 943.1305972297828
	agregadofino: 751.2503230214093
	idade: 338.15931880218807


In [ ]:
target_desejado = 50.0
problem = ReceitaPorResistencia(model, target_desejado)
res = minimize(problem, algorithm, ('n_gen', 100), seed=42)
melhor_solucao = res.X[0, :].reshape(1, -1)

print(f"Alvo: {target_desejado} MPa")
print(f"Ingredientes para atingir o alvo: \n\t{"\n\t".join([f'{name}: {value}' for name, value in zip(feature_names, melhor_solucao.flatten())])}")
print(f"Resistência da Receita Encontrada: {model.predict(melhor_solucao)[0]:.2f} MPa")
print(f"Erro: {res.F[0][0]:.4f}")

Alvo: 50.0 MPa
Ingredientes para atingir o alvo: 
	cimento: 234.46725201583416
	escoriadealtoforno: 183.84021133734393
	cinzasvolantes: 149.4293370749516
	agua: 241.23721072272045
	superplastificante: 13.145395535749577
	agregadogrosso: 890.7695298835864
	agregadofino: 948.8145630728084
	idade: 243.5113557453538
Resistência da Receita Encontrada: 50.00 MPa
Erro: 0.0004
